# Single- versus double-precision dense cuboid direct

This simple notebook compares reusable FP32 and FP64 static tensor plans for the same cuboid geometry. It reports construction time, static tensor memory, repeated evaluation time, and field error. Geometry is evaluated robustly before the completed six-component tensors are stored in the selected precision.


## Human-readable problem setup

A modest cuboid lattice keeps the notebook quick to run. The non-uniform magnetisation makes every tensor component participate in the result.


In [ ]:
import time

import cdfmm
import matplotlib.pyplot as plt
import numpy as np

SIDE = 10.0e-9
SPACING = 3.0 * SIDE
SHAPE = (8, 8, 8)
REPEATS = 10

indices = np.indices(SHAPE, dtype=np.float64).reshape(3, -1).T
centres = (indices - (np.asarray(SHAPE) - 1) / 2) * SPACING
particle_number = np.arange(len(centres), dtype=np.float64)
magnetisation = np.column_stack((
    7.0e5 + 1.5e5 * np.sin(0.17 * particle_number),
    -3.0e5 + 2.0e5 * np.cos(0.11 * particle_number),
    4.0e5 * np.sin(0.07 * particle_number + 0.3),
))
moments = SIDE**3 * magnetisation
cube = cdfmm.CuboidSize(SIDE, SIDE, SIDE)

print(f"Cuboids: {len(centres):,}")


## Construct the two reusable plans

Only `static_precision` changes. FP64 remains the default, but it is written explicitly here so the comparison is easy to read.


In [ ]:
def construct_plan(static_precision):
    start = time.perf_counter()
    plan = cdfmm.DenseDirectPlan(
        source_positions=centres,
        target_positions=centres,
        source_geometry=cdfmm.SourceGeometry.UNIFORM_CUBOID,
        target_geometry=cdfmm.TargetGeometry.POINT,
        source_sizes=[cube],
        static_precision=static_precision,
    )
    elapsed = time.perf_counter() - start
    return plan, elapsed


fp64_plan, fp64_construction_s = construct_plan("float64")
fp32_plan, fp32_construction_s = construct_plan("float32")

print(f"FP64 construction: {fp64_construction_s:.6f} s")
print(f"FP32 construction: {fp32_construction_s:.6f} s")
print(f"FP64 tensors:      {fp64_plan.tensor_memory_bytes / 2**20:.3f} MiB")
print(f"FP32 tensors:      {fp32_plan.tensor_memory_bytes / 2**20:.3f} MiB")
print(f"Memory ratio:      {fp32_plan.tensor_memory_bytes / fp64_plan.tensor_memory_bytes:.3f}")


## Repeated evaluation runtime

One untimed call warms each path. Median runtime reduces sensitivity to unrelated system activity. The portable backend makes the comparison available without oneMKL.


In [ ]:
def evaluate_repeatedly(plan, input_moments):
    plan.evaluate(input_moments, backend=cdfmm.DenseDirectBackend.PORTABLE)
    samples = []
    for _ in range(REPEATS):
        start = time.perf_counter()
        field = plan.evaluate(
            input_moments,
            backend=cdfmm.DenseDirectBackend.PORTABLE,
        )
        samples.append(time.perf_counter() - start)
    return field, np.asarray(samples)


H_fp64, fp64_samples = evaluate_repeatedly(fp64_plan, moments)
H_fp32, fp32_samples = evaluate_repeatedly(fp32_plan, moments)
fp64_evaluation_s = float(np.median(fp64_samples))
fp32_evaluation_s = float(np.median(fp32_samples))

print(f"FP64 median evaluation: {fp64_evaluation_s:.6f} s")
print(f"FP32 median evaluation: {fp32_evaluation_s:.6f} s")
print(f"FP64 / FP32 speed ratio: {fp64_evaluation_s / fp32_evaluation_s:.3f}x")


## Precision loss relative to FP64

The global relative L2 error measures the complete field, while the maximum pointwise relative error excludes effectively zero reference fields.


In [ ]:
difference = H_fp32 - H_fp64
reference_norms = np.linalg.norm(H_fp64, axis=1)
absolute_errors = np.linalg.norm(difference, axis=1)
meaningful = reference_norms > 1.0e-12 * reference_norms.max()

relative_l2 = np.linalg.norm(difference) / np.linalg.norm(H_fp64)
maximum_absolute = absolute_errors.max()
maximum_pointwise_relative = np.max(
    absolute_errors[meaningful] / reference_norms[meaningful]
)

print(f"Relative L2 error:               {relative_l2:.3e}")
print(f"Maximum absolute field error:   {maximum_absolute:.3e} A/m")
print(f"Maximum pointwise relative error: {maximum_pointwise_relative:.3e}")


## Compact comparison plot


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels = ["FP64", "FP32"]
colours = ["C0", "C1"]

axes[0].bar(labels, [fp64_plan.tensor_memory_bytes / 2**20,
                     fp32_plan.tensor_memory_bytes / 2**20], color=colours)
axes[0].set(ylabel="Static tensor memory [MiB]", title="Plan storage")

axes[1].bar(labels, [fp64_evaluation_s * 1e3, fp32_evaluation_s * 1e3],
            color=colours)
axes[1].set(ylabel="Median evaluation [ms]", title="Repeated runtime")

axes[2].hist(np.maximum(absolute_errors, np.finfo(float).tiny), bins=30,
             color="C1")
axes[2].set_xscale("log")
axes[2].set(xlabel="|H32 - H64| [A/m]", ylabel="Targets",
            title="FP32 field error")

for axis in axes:
    axis.grid(alpha=0.3)
fig.tight_layout()
plt.show()


## Interpretation and scope

The FP32 plan stores exactly half as many tensor bytes and executes its portable matrix-vector products in FP32. FP64 input moments are converted into reusable FP32 staging arrays for FP32 evaluation; callers may also pass NumPy `float32` moments. Results are returned as NumPy float64 arrays at the public boundary.

This notebook measures the CPU dense-direct implementation. Uniform-FMM and CUDA static plans remain FP64 in the current implementation, so the notebook deliberately does not present an unsupported FP32 tree comparison.
